In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import randint

In [ ]:
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

In [ ]:
X.isnull().sum()

In [ ]:
X.info()

In [ ]:
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
dt_regressor = DecisionTreeRegressor(random_state=42)

In [ ]:
dt_regressor.fit(X_train, y_train)

In [ ]:
y_pred = dt_regressor.predict(X_test)

In [ ]:
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

In [ ]:
print(f"Initial Model Mean Squared Error: {mse:.4f}")
print(f"Initial Model R-squared Score: {r2:.4f}")

In [ ]:
param_dist = {
    'max_depth': randint(3, 15),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 20),
    'max_features': ['auto', 'sqrt', 'log2', None]
}

In [ ]:
random_search = RandomizedSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=50,
    cv=5,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

In [ ]:
random_search.fit(X_train, y_train)

In [ ]:
print("\nBest Parameters from RandomizedSearchCV:", random_search.best_params_)

In [ ]:
best_model = random_search.best_estimator_

In [ ]:
y_pred_best = best_model.predict(X_test)

In [ ]:
print(f"Best Model MSE: {mean_squared_error(y_test, y_pred_best):.4f}")
print(f"Best Model R²: {r2_score(y_test, y_pred_best):.4f}")

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

In [ ]:
print("\nFeature Importance:\n", feature_importance)

In [ ]:
import joblib

In [ ]:
joblib.dump(best_model, 'decision_tree_california_randomsearch_model.pkl')

In [ ]:
from sklearn.pipeline import Pipeline

In [ ]:
pipeline = Pipeline([
    ('decision_tree', DecisionTreeRegressor(**random_search.best_params_, random_state=42))
])

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
y_pred_pipeline = pipeline.predict(X_test)

In [ ]:
print(f"\nPipeline MSE: {mean_squared_error(y_test, y_pred_pipeline):.4f}")
print(f"Pipeline R²: {r2_score(y_test, y_pred_pipeline):.4f}")